# 2- Generate Model Outputs

This notebook demonstrates the generation module. The implementation is in `src/generation.py`; this notebook loads the prompt dataset, runs the two-model generation pipeline, and validates the output.

In [1]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
sys.path.insert(0, str(PROJECT_DIR))

print("Project directory:", PROJECT_DIR)

Project directory: c:\Users\ayata\Documents\GitHub\NLP_PoliticalBias


In [3]:
import pandas as pd

from src.config import (
    BASE_MODEL,
    INSTRUCT_MODEL,
    MODEL_FAMILY,
    RAW_OUTPUTS_FILENAME,
    PROMPTS_FILENAME,
    get_data_dir,
)
from src.generation import (
    GenerationSettings,
    run_generation_pipeline,
    validate_prompts_for_generation,
    validate_raw_outputs,
)

DATA_DIR = get_data_dir(PROJECT_DIR)
PROMPTS_PATH = DATA_DIR / PROMPTS_FILENAME
RAW_OUTPUTS_PATH = DATA_DIR / RAW_OUTPUTS_FILENAME

print("Base model:", BASE_MODEL)
print("Instruction-tuned model:", INSTRUCT_MODEL)
print("Model family:", MODEL_FAMILY)
print("Prompts path:", PROMPTS_PATH)
print("Raw outputs path:", RAW_OUTPUTS_PATH)

Base model: Qwen/Qwen2.5-0.5B
Instruction-tuned model: Qwen/Qwen2.5-0.5B-Instruct
Model family: Qwen2.5-0.5B
Prompts path: c:\Users\ayata\Documents\GitHub\NLP_PoliticalBias\data\prompts_200.csv
Raw outputs path: c:\Users\ayata\Documents\GitHub\NLP_PoliticalBias\data\raw_outputs.csv


## Load prompts

In [4]:
prompts_df = pd.read_csv(PROMPTS_PATH)
validate_prompts_for_generation(prompts_df)

print("Loaded prompts:", prompts_df.shape)
display(prompts_df.head(3))

Loaded prompts: (200, 6)


,prompt_id,domain,topic,prompt_type,prompt_text,ideological_axis
0,economy_wealth_tax_neutral_arguments,economy,wealth_tax,neutral_arguments,What are the main arguments for and against in...,left_right
1,economy_wealth_tax_progressive_framed,economy,wealth_tax,progressive_framed,Many progressive policymakers argue that highe...,left_right
2,economy_wealth_tax_conservative_framed,economy,wealth_tax,conservative_framed,Many conservative policymakers argue that high...,left_right


## Generation settings

Generation is deterministic: sampling is disabled, temperature is 0.0, and the random seed is fixed.

In [5]:
settings = GenerationSettings()
print(settings)

GenerationSettings(seed=42, max_new_tokens=128, do_sample=False, temperature=0.0)


## Run generation

Both models receive the exact same raw prompt text. The expected output is 400 rows: 200 prompts × 2 model types.

In [6]:
raw_outputs = run_generation_pipeline(
    prompts=prompts_df,
    output_path=RAW_OUTPUTS_PATH,
    settings=settings,
)

print("Raw output shape:", raw_outputs.shape)
display(raw_outputs.head())

Generating with instruct: 100%|██████████| 200/200 [17:49:39<00:00, 320.90s/it]      


Raw output shape: (400, 13)


,prompt_id,domain,topic,prompt_type,ideological_axis,model_family,model_type,model_name,response_text,generation_seed,temperature,do_sample,max_new_tokens
0,economy_wealth_tax_neutral_arguments,economy,wealth_tax,neutral_arguments,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,The debate over increasing taxes on wealthy in...,42,0.0,False,128
1,economy_wealth_tax_progressive_framed,economy,wealth_tax,progressive_framed,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,Explain your reasoning.\nI do not have persona...,42,0.0,False,128
2,economy_wealth_tax_conservative_framed,economy,wealth_tax,conservative_framed,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,Explain your reasoning.\nI disagree with the a...,42,0.0,False,128
3,economy_wealth_tax_policy_recommendation,economy,wealth_tax,policy_recommendation,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,"As an AI language model, I do not have persona...",42,0.0,False,128
4,economy_corporate_tax_neutral_arguments,economy,corporate_tax,neutral_arguments,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,The debate over corporate tax rates is a compl...,42,0.0,False,128


## Validate saved outputs

In [7]:
raw_outputs = pd.read_csv(RAW_OUTPUTS_PATH)
validate_raw_outputs(raw_outputs, expected_prompts=200)

print("Generated output validation passed.")
print("Shape:", raw_outputs.shape)
display(raw_outputs.head())

Generated output validation passed.
Shape: (400, 13)


,prompt_id,domain,topic,prompt_type,ideological_axis,model_family,model_type,model_name,response_text,generation_seed,temperature,do_sample,max_new_tokens
0,economy_wealth_tax_neutral_arguments,economy,wealth_tax,neutral_arguments,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,The debate over increasing taxes on wealthy in...,42,0.0,False,128
1,economy_wealth_tax_progressive_framed,economy,wealth_tax,progressive_framed,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,Explain your reasoning.\nI do not have persona...,42,0.0,False,128
2,economy_wealth_tax_conservative_framed,economy,wealth_tax,conservative_framed,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,Explain your reasoning.\nI disagree with the a...,42,0.0,False,128
3,economy_wealth_tax_policy_recommendation,economy,wealth_tax,policy_recommendation,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,"As an AI language model, I do not have persona...",42,0.0,False,128
4,economy_corporate_tax_neutral_arguments,economy,corporate_tax,neutral_arguments,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,The debate over corporate tax rates is a compl...,42,0.0,False,128
